# Smart WhatsApp Agent — Teste interativo do RAG

### O que este notebook faz?
- Carrega a base de conhecimento (52 chunks)
- Indexa no ChromaDB (embedding local, sem custo)
- **Você digita uma frase** e vê quais chunks foram recuperados e a **categoria** de cada um

> ⚠️ Importante: isto é **busca (retrieval)**, não classificação de sentimento. Mostra *qual conhecimento* combina com a sua frase.

## 1. Preparação (rode uma vez)

Confere se as bibliotecas estão instaladas no Python que você está usando.

In [1]:
import sys, subprocess

try:
    import chromadb, sentence_transformers  # noqa: F401
    print("Bibliotecas OK")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "chromadb", "sentence-transformers"])
    print("Instaladas")

Bibliotecas OK


## 2. Importar nosso código

> 🔑 **Importante:** esta célula adiciona a pasta do notebook ao `sys.path` automaticamente. Assim funciona mesmo que o Jupyter tenha sido iniciado de outra pasta.

In [2]:
import sys, os

# Garante que a pasta do projeto (onde fica a pasta 'rag/') esteja no path
PASTA_PROJETO = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()

if PASTA_PROJETO not in sys.path:
    sys.path.insert(0, PASTA_PROJETO)

os.chdir(PASTA_PROJETO)  # garante que 'data/...' seja resolvido na pasta certa

print("Pasta do projeto:", PASTA_PROJETO)
print("Conteúdo da pasta:", [d for d in os.listdir(PASTA_PROJETO) if not d.startswith('.')])

Pasta do projeto: c:\Users\joth1\Desktop\Project-AVA-BOT-Whatsapp\smart-whatsapp-agent
Conteúdo da pasta: ['chroma_db', 'data', 'Documentação Projeto.docx', 'inspector.py', 'Manual_Institucional_FitLife_Academia_v2_0.docx', 'rag', 'README.md', 'requirements.txt', 'requisitos_sistema.xlsx', 'teste_rag.ipynb', 'tests', '__pycache__']


In [3]:
from rag.retriever import buscar
from rag.indexer import indexar_base
print("Imports OK")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Imports OK


## 3. Indexar a base (rode uma vez)

Cria o `chroma_db` com os 52 chunks. Leva alguns segundos na primeira vez.

In [4]:
total = indexar_base()
print(f"Base indexada: {total} chunks")

Base indexada: 52 chunks


## 4. Teste sua frase ✍️

**Edite a caixa abaixo** com qualquer pergunta sobre a academia e rode a célula.

In [5]:
frase = "Quanto custa o plano anual?"  # <- EDITE AQUI e rode
top_k = 3

resultado = buscar(frase, top_k=top_k)

ids = resultado["ids"][0]
metadados = resultado["metadatas"][0]
distancias = resultado["distances"][0]

# Define a mensagem inicial
print("=" * 60)
print(f"FRASE: {frase}")
print("=" * 60)



for i in range(len(ids)):
    m = metadados[i]
    sim = 1 - distancias[i]
    print(f"\nRank {i+1} | {ids[i]} | confiança {sim*100:.0f}%")
    print(f"  \u2705 Isto se refere a:  {m.get('categoria')} -> {m.get('titulo')}")
    print(f"  Conteúdo  : {m.get('conteudo')}")

frase = "Quanto custa o plano anual?"  # <- EDITE AQUI e rode
top_k = 3

resultado = buscar(frase, top_k=top_k)

ids = resultado["ids"][0]
metadados = resultado["metadatas"][0]
distancias = resultado["distances"][0]

# Define a mensagem inicial
print("=" * 60)
print(f"FRASE: {frase}")
print("=" * 60)



for i in range(len(ids)):
    m = metadados[i]
    sim = 1 - distancias[i]
    print(f"\nRank {i+1} | {ids[i]} | confiança {sim*100:.0f}%")
    print(f"  \u2705 Isto se refere a:  {m.get('categoria')} -> {m.get('titulo')}")
    print(f"  Conteúdo  : {m.get('conteudo')}")    

FRASE: Quanto custa o plano anual?

Rank 1 | FIT-PLAN-003 | confiança 69%
  ✅ Isto se refere a:  Planos -> Plano Anual
  Conteúdo  : Plano anual no valor de R$69 por mês com fidelidade de 12 meses.

Rank 2 | FIT-PLAN-002 | confiança 66%
  ✅ Isto se refere a:  Planos -> Plano Trimestral
  Conteúdo  : Plano trimestral no valor de R$89 por mês.

Rank 3 | FIT-PLAN-001 | confiança 51%
  ✅ Isto se refere a:  Planos -> Plano Mensal
  Conteúdo  : Plano mensal no valor de R$99 sem fidelidade.
FRASE: Quanto custa o plano anual?

Rank 1 | FIT-PLAN-003 | confiança 69%
  ✅ Isto se refere a:  Planos -> Plano Anual
  Conteúdo  : Plano anual no valor de R$69 por mês com fidelidade de 12 meses.

Rank 2 | FIT-PLAN-002 | confiança 66%
  ✅ Isto se refere a:  Planos -> Plano Trimestral
  Conteúdo  : Plano trimestral no valor de R$89 por mês.

Rank 3 | FIT-PLAN-001 | confiança 51%
  ✅ Isto se refere a:  Planos -> Plano Mensal
  Conteúdo  : Plano mensal no valor de R$99 sem fidelidade.


## 4b. Ver todos os 52 chunks da base

A base tem **52 chunks** em **12 categorias**. Esta célula lista todos para você conferir.

In [6]:
from rag.loader import carregar_base
from collections import defaultdict

base = carregar_base()
chunks = base["chunks"]

print(f"TOTAL DE CHUNKS: {len(chunks)}\n")

por_categoria = defaultdict(list)
for c in chunks:
    por_categoria[c["categoria"]].append((c["chunk_id"], c["titulo"]))

for categoria in sorted(por_categoria):
    print(f"### {categoria} ({len(por_categoria[categoria])})")
    for cid, titulo in por_categoria[categoria]:
        print(f"   {cid}  ->  {titulo}")
    print()

TOTAL DE CHUNKS: 52

### Atendimento Especializado (5)
   FIT-ATE-001  ->  Acessibilidade e Necessidades Especiais
   FIT-ATE-002  ->  Agendamento de Aulas
   FIT-ATE-003  ->  Horários de Maior Movimento
   FIT-ATE-004  ->  Objetos Perdidos
   FIT-ATE-005  ->  Acompanhantes e Convidados

### FAQ (7)
   FIT-FAQ-001  ->  Como realizar matrícula
   FIT-FAQ-002  ->  Aula Experimental
   FIT-FAQ-003  ->  Documentos Necessários
   FIT-FAQ-004  ->  Taxa de Matrícula
   FIT-FAQ-005  ->  Treinar em Qualquer Horário
   FIT-FAQ-006  ->  Estacionamento
   FIT-FAQ-007  ->  Canais de Atendimento

### Grade de Aulas (4)
   FIT-AULA-001  ->  Funcional - Horário
   FIT-AULA-002  ->  HIIT - Horário
   FIT-AULA-003  ->  Alongamento - Horário
   FIT-AULA-004  ->  Pilates Solo - Horário

### Horários (2)
   FIT-HOR-001  ->  Horário de Funcionamento
   FIT-HOR-002  ->  Horários de Feriados

### Informações Gerais (11)
   FIT-INFO-001  ->  Formas de Pagamento
   FIT-INFO-002  ->  Idade Mínima para Treinar
  

## 4c. Explorar os 52 chunks em detalhes (acordeões)

A mesma base, porém **navegável**: cada chunk é um bloco expansível mostrando **todos** os metadados
(id, título, categoria, tipo de conhecimento, prioridade, status, tags, relacionamentos e conteúdo).
Clique para abrir cada um. Rode a célula abaixo e veja o resultado.

In [7]:
import ipywidgets as widgets
from IPython.display import display
import html
from rag.loader import carregar_base

base = carregar_base()

campos = ["titulo", "categoria", "tipo_conhecimento", "status", "prioridade",
          "documento", "versao", "responsavel", "ultima_atualizacao", "tags",
          "relacionamentos", "conteudo"]

def montar_linha(chave, valor):
    if valor in (None, ""):
        return ""
    return (f"<tr><td style='width:180px;color:#888;vertical-align:top'>"
            f"{html.escape(chave)}</td><td>{html.escape(str(valor))}</td></tr>")

cards = []
for chunk in base["chunks"]:
    titulo = chunk.get("titulo", "")
    linha = "".join(montar_linha(c, chunk.get(c)) for c in campos)
    corpo = widgets.HTML(f"<table style='font-size:13px'>{linha}</table>")
    acordeao = widgets.Accordion(children=[corpo])
    acordeao.set_title(0, "Metadados + conteudo")
    acordeao.selected_index = None
    cabecalho = widgets.HTML(
        f"<b style='font-size:16px'>{html.escape(chunk['chunk_id'])}</b> &nbsp; "
        f"<span style='font-size:14px'>{html.escape(titulo)}</span>"
    )
    cards.append(widgets.VBox([cabecalho, acordeao]))

print(f"Total de chunks: {len(cards)}")
print("Cada card e expansivel. Clique em 'Metadados + conteudo' para abrir.\n")
# for c in cards:
    # display(c)


Total de chunks: 52
Cada card e expansivel. Clique em 'Metadados + conteudo' para abrir.



# 5. Teste Oficial — Critério de Aceite

Esta etapa valida o MVP do RAG executando as **5 perguntas oficiais do projeto**, verificando se o chunk esperado aparece entre os **Top-3 resultados** da busca vetorial.

> **Critério de aprovação:** 5/5 perguntas recuperando o chunk correto entre os Top-3.

## Base de Conhecimento

**Total de Chunks:** **52**

---

## FAQ (7 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-FAQ-001` | Como realizar matrícula |
| `FIT-FAQ-002` | Aula Experimental |
| `FIT-FAQ-003` | Documentos Necessários |
| `FIT-FAQ-004` | Taxa de Matrícula |
| `FIT-FAQ-005` | Treinar em Qualquer Horário |
| `FIT-FAQ-006` | Estacionamento |
| `FIT-FAQ-007` | Canais de Atendimento |

---

## Planos (4 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-PLAN-001` | Plano Mensal |
| `FIT-PLAN-002` | Plano Trimestral |
| `FIT-PLAN-003` | Plano Anual |
| `FIT-PLAN-004` | Processo de Inscrição |

---

## Localização (3 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-LOC-001` | Endereço e Localização da FitLife |
| `FIT-LOC-002` | Como Chegar à FitLife |
| `FIT-LOC-003` | Estacionamento da FitLife |

---

## Horários (2 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-HOR-001` | Horário de Funcionamento |
| `FIT-HOR-002` | Horários de Feriados |

---

## Política de Cancelamento e Plano (4 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-POL-001` | Cancelamento Plano Mensal |
| `FIT-POL-002` | Cancelamento Plano Anual |
| `FIT-POL-003` | Procedimento de Cancelamento |
| `FIT-POL-004` | Transferência de Plano |

---

## Atendimento Especializado (5 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-ATE-001` | Acessibilidade e Necessidades Especiais |
| `FIT-ATE-002` | Agendamento de Aulas |
| `FIT-ATE-003` | Horários de Maior Movimento |
| `FIT-ATE-004` | Objetos Perdidos |
| `FIT-ATE-005` | Acompanhantes e Convidados |

---

## Informações Gerais (11 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-INFO-001` | Formas de Pagamento |
| `FIT-INFO-002` | Idade Mínima para Treinar |
| `FIT-INFO-003` | Estrutura e Vestiários |
| `FIT-INFO-004` | Saúde e Atestado Médico |
| `FIT-INFO-005` | Congelamento de Plano |
| `FIT-INFO-006` | Unidade da FitLife |
| `FIT-INFO-007` | Convênios e Benefícios Corporativos |
| `FIT-INFO-008` | Atualização de Dados Cadastrais |
| `FIT-INFO-009` | Comprovantes e Nota Fiscal |
| `FIT-INFO-010` | Descontos e Promoções |
| `FIT-INFO-011` | Fotos e Vídeos na Academia |

---

## Regulamento (3 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-REG-001` | Uso dos Equipamentos |
| `FIT-REG-002` | Regras de Segurança |
| `FIT-REG-003` | Convivência |

---

## Produtos (3 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-SRV-001` | Whey Protein |
| `FIT-SRV-002` | Creatina |
| `FIT-SRV-003` | Garrafa FitLife |

---

## Serviços (2 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-SRV-004` | Personal Trainer |
| `FIT-SRV-005` | Avaliação Física |

---

## Modalidades (4 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-MOD-001` | Musculação |
| `FIT-MOD-002` | Funcional |
| `FIT-MOD-003` | HIIT |
| `FIT-MOD-004` | Pilates Solo |

---

## Grade de Aulas (4 Chunks)

| Chunk ID | Descrição |
|----------|-----------|
| `FIT-AULA-001` | Funcional – Horário |
| `FIT-AULA-002` | HIIT – Horário |
| `FIT-AULA-003` | Alongamento – Horário |
| `FIT-AULA-004` | Pilates Solo – Horário |

---

## Critério Oficial de Validação

Durante a execução dos testes, o sistema deverá responder corretamente às seguintes perguntas, recuperando o chunk esperado entre os **Top-3 resultados**.

| Pergunta | Chunk Esperado |
|----------|----------------|
| Quanto custa o plano anual? | `FIT-PLAN-003` |
| Posso cancelar meu plano? | `FIT-POL-002` |
| Que horas abre domingo? | `FIT-HOR-001` |
| Tem funcional? | `FIT-MOD-002` |
| Posso fazer aula teste? | `FIT-FAQ-002` |

### Resultado Esperado

- [ ] 5/5 perguntas recuperando o chunk correto.
- [ ] Similaridade registrada para cada resultado.
- [ ] Exibição do Top-3 no Retrieval Inspector.
- [ ] Contexto enviado ao LLM registrado para auditoria.

In [8]:
import importlib
import tests.test_5_questoes
importlib.reload(tests.test_5_questoes)  # garante a versao mais nova do modulo
from tests.test_5_questoes import executar
ok = executar(imprimir=True)
print("APROVADO" if ok else "REPROVADO")


Pergunta: Quanto custa o plano anual?
  -> Esperado  : FIT-PLAN-003 [Planos - Valor do plano anual]
  -> Resultado : APROVADO (rank 1 no Top-3)
  -> Os 3 chunks recuperados (mais relevantes primeiro):
      • FIT-PLAN-003 [Planos] Plano Anual
      • FIT-PLAN-002 [Planos] Plano Trimestral
      • FIT-PLAN-001 [Planos] Plano Mensal
Pergunta: Posso cancelar meu plano?
  -> Esperado  : FIT-POL-002 [Politica - Cancelamento do plano anual]
  -> Resultado : APROVADO (rank 2 no Top-3)
  -> Os 3 chunks recuperados (mais relevantes primeiro):
      • FIT-POL-001 [Política] Cancelamento Plano Mensal
      • FIT-POL-002 [Política] Cancelamento Plano Anual
      • FIT-INFO-005 [Informações Gerais] Congelamento de Plano
Pergunta: Que horas abre domingo?
  -> Esperado  : FIT-HOR-001 [Horarios - Abertura aos domingos]
  -> Resultado : APROVADO (rank 1 no Top-3)
  -> Os 3 chunks recuperados (mais relevantes primeiro):
      • FIT-HOR-001 [Horários] Horário de Funcionamento
      • FIT-AULA-001 [Grade 

## 5b. Passar sua própria lista de perguntas 💬

A função `executar` aceita listas/dicts e adapta o comportamento:

- `executar()` → usa as 5 oficiais (validação)
- `executar(["sua frase"])` → **modo livre** (só mostra o que a busca acha)
- `executar({"frase": "CHUNK"})` → **validação** com a sua expectativa

Rode as células abaixo e observe a saída.


In [9]:
import importlib
import tests.test_5_questoes
importlib.reload(tests.test_5_questoes)  # garante a versao mais nova do modulo
from tests.test_5_questoes import executar
# MODO LIVRE: passo so as perguntas (sem chunk esperado); a funcao apenas
# busca e mostra. Edite a lista como quiser.
minhas_perguntas = [
    "Qual o horario da aula de dança?",
    "Como posso realizar minha inscrição?",
    # "Qual o endereço da academia?",
    # "Vocês possuem algum aplicativo da academia ?",
    "Perdi um item na academia, podem verificar ?"
]
ok = executar(minhas_perguntas, imprimir=True)

Pergunta: Qual o horario da aula de dança?
  -> Melhores chunks recuperados (mais relevantes primeiro):
      • FIT-AULA-003 [Grade de Aulas] Alongamento - Horário
      • FIT-AULA-001 [Grade de Aulas] Funcional - Horário
      • FIT-AULA-004 [Grade de Aulas] Pilates Solo - Horário
Pergunta: Como posso realizar minha inscrição?
  -> Melhores chunks recuperados (mais relevantes primeiro):
      • FIT-PLAN-004 [Planos] Processo de Inscrição
      • FIT-FAQ-001 [FAQ] Como realizar matrícula
      • FIT-POL-004 [Política] Transferência de Plano
Pergunta: Perdi um item na academia, podem verificar ?
  -> Melhores chunks recuperados (mais relevantes primeiro):
      • FIT-ATE-004 [Atendimento Especializado] Objetos Perdidos
      • FIT-LOC-001 [Localização] Endereço e Localização da FitLife
      • FIT-AULA-001 [Grade de Aulas] Funcional - Horário

Modo livre: nenhuma pergunta tinha chunk esperado — apenas exploracao.


# 6. Sprint 5 — Resposta com LLM local (Llama 3.2 1B)
Pipeline completo: **pergunta → RAG (recupera o chunk) → Llama (percebe o sentimento e gera a resposta no tom adequado)**.

> ⚠️ O modelo baixa ~2,5 GB na primeira execução. Depois funciona 100% offline.
> ⚠️ Na CPU, cada resposta leva ~15-30 s. (Testado de verdade aqui.)

In [1]:
import importlib
import llm.responder
importlib.reload(llm.responder)  # garante a versao mais nova do codigo
from llm.responder import responder

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# EDITE a pergunta abaixo e rode. Observe que o Llama:
#  - usa o chunk recuperado pelo RAG como base da resposta;
#  - percebe o tom/sentimento do cliente e ajusta a resposta.
pergunta = "Quero cancelar meu plano AGORA, estou muito insatisfeito!"
resposta = responder(pergunta)
print("Pergunta:", pergunta)
print("Resposta:", resposta)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pergunta: Quero cancelar meu plano AGORA, estou muito insatisfeito!
Resposta: Entendo sua insatisfação, mas gostaria de ajudá-lo a entender os termos do plano. O plano mensal pode ser cancelado a qualquer momento sem multa após o período vigente. Se estiver insatisfeito, pode considerar a opção de cancelar o plano anual, que possui fidelidade mínima de 12 meses. Se precisar de mais informações, por favor, não hesite em entrar em contato.


In [5]:
# Compare: a mesma pergunta com TONS opostos.
exemplos = [
    # "Quanto custa o plano anual?",
    # "REVOLTADO! Vou processar voces e reclamar em todas as redes!",
    "Não aguento mais implorar, por favor, cancelem meu plano"
]
for p in exemplos:
    print("=" * 70)
    print("PERGUNTA:", p)
    print("RESPOSTA:", responder(p))
print("=" * 70)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PERGUNTA: Não aguento mais implorar, por favor, cancelem meu plano
RESPOSTA: Entendo sua frustração, mas gostaria de ajudar a resolver essa situação de forma clara. Você está certa de que não está cumprindo o plano mínimo de 12 meses para cancelá-lo. Vamos verificar a situação e agendar uma hora para discutir a possibilidade de cancelamento. Por favor, compartilhe mais detalhes para que possamos agendar uma consulta.
